# 15 · Stage B (thu gọn) — Disease gate + **global WASL α=.50**

## Vì sao chạy dù Stage A trượt cổng — deviation được ghi rõ

Stage A **không** đạt ngưỡng preregistered (`G_DA_macro +0.0011`, `G_DA_R3 +0.0007`, cả hai
< +0.002 tại checkpoint được chọn theo §11). Chạy tiếp là **chệch khỏi prereg**, dựa trên ba
căn cứ, ghi lại đây để không bị đọc nhầm về sau:

1. **Correspondence dương và mạnh.** `REAL − SHUFFLE macro = +0.0028` lớn hơn
   `REAL − NULL = +0.0012` ⇒ EHR của **bệnh nhân khác tệ hơn không có EHR** (0.0016).
   Model điều kiện hoá theo **nội dung**, không dùng EHR như hằng số hay capacity thừa.
2. **Bootstrap paired loại trừ 0**: `Rare-3 +0.0007 [+0.0001, +0.0013]`. Hiệu ứng là thật,
   chỉ là **nhỏ hơn thanh xà đã đặt** — cận trên CI (+0.0013) vẫn dưới +0.002.
3. **Artifact chọn checkpoint.** `G_DA_R3` theo epoch tăng đơn điệu tới **+0.0023 @ep5**,
   nhưng §11 bốc **ep2** (macro đỉnh sớm) nơi nó chỉ +0.0007. Gate phát triển muộn:
   `A1` res/z `.0530→.0578→.0668→.0805`, gate `.252→.263→.273→.302`, trong khi `A0`
   res/z tắt ngóm `.0001`.

**Giới hạn phải ghi vào báo cáo**: đây là kết quả 1 seed, chệch prereg, và độ lớn hiệu ứng
của gate nhỏ hơn ngưỡng thực dụng đã khoá.

## Vì sao **B1** chứ không phải B2/B3

`B1` = WASL toàn cục, gradient đi qua **toàn mạng** ⇒ cơ sở của nó **độc lập với việc gate
mạnh hay yếu**. Câu hỏi nó trả lời: kiến trúc mới có `L_IMG` + L2-SP **bảo vệ nhánh ảnh** —
có chữa được cú sập của WASL .50 ở nb12 không (tail `0.1230@ep2 → 0.1098@ep8`, sụt `0.0132`)?

`B2`/`B3` định tuyến phần nhấn thêm **chỉ** qua nhánh residual đã cô lập — mà Stage A vừa đo
được nhánh đó chỉ dịch R3 khoảng `+0.0007` (CI ≤ +0.0013). Trần của B2/B3 bị chặn bởi chính
con số đó. Hoãn lại, chỉ chạy nếu B1 cho thấy trần này không phải nút thắt.

## Arm

| arm | EHR | loss |
|---|---|---|
| `B1` | thật (DA-E) | `WASL.50(z^MM) + 0.25·ASL(z^IMG)` |
| `B1u` | NULL (DA-U) | `WASL.50(z^MM) + 0.25·ASL(z^IMG)` |

**`B0` không train lại** — nó trùng khít cấu hình `A1` của nb14 (DA-E + ASL, cùng seed, cùng
init, cùng thứ tự batch). Đọc thẳng từ `disease_gate/metrics/epoch_log.json`, tiết kiệm 1/3
compute. `HY_BATCH` bị **ép bằng** `BATCH` của nb14, nếu lệch thì assert dừng.

`B1u` là **control giai thừa** — không có nó thì `B1` dương không quy được cho gate hay cho
kiến trúc image-fallback.

## Estimand

```
Delta_WASL      = B1  - A1        (doi ASL -> WASL trong cung kien truc, paired)
G_DA | WASL     = B1  - B1u       (hieu ung noi dung EHR DUOI WASL)
G_DA | ASL      = A1  - A0        (da co tu nb14)
I_loss x EHR    = (B1 - B1u) - (A1 - A0)
```

`I_loss×EHR > 0` nghĩa là: **EHR hữu ích hơn khi gradient nhãn hiếm đủ mạnh để pathway fusion
học cách khai thác nó** — đúng Case D của `FINAL_PLAN_WEIGHTED_DEFERRED_ASL_WITH_METRICS.md` §20.

## Stability — câu hỏi trung tâm (`REVISION` §9)

So `peak / epoch of peak / ep8 / peak-drop` của `B1` với WASL toàn cục của nb12.
Thành công = **gain Rare-3 tương đương hoặc hơn, nhưng sụt peak→ep8 nhỏ hơn rõ rệt**.

## C0 · Config + preregistration (Stage B thu gọn)

In [ ]:
import os, json, math, time, copy, pickle, random, shutil, tarfile, contextlib
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
LOCAL_IMG = os.environ.get('LOCAL_IMG', '/content/cxr_local')
SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')
H5_CKPT   = os.path.join(DATA_DIR, 'token_fusion', 'checkpoints', 'H5_s42_best.pt')
NB14_DIR  = os.path.join(DATA_DIR, 'disease_gate')      # A0/A1 lam moc paired
NB12_DIR  = os.path.join(DATA_DIR, 'loss_screen')       # WASL .50 toan cuc (kien truc CU)
SC_DIR    = os.path.join(DATA_DIR, 'disease_gate_wasl')
SNAP_DIR  = f'{SC_DIR}/snapshots'
for _d in (SC_DIR, f'{SC_DIR}/checkpoints', f'{SC_DIR}/metrics', SNAP_DIR): os.makedirs(_d, exist_ok=True)

W_F, W_L = 0.7, 0.3
SEED     = 42

# --- protocol: BAT BUOC y het nb14 (chi doi loss) ---
DECODER_WARMUP_EPOCHS = 1
LR_ADAPTER, LR_DECODER = 1e-4, 1e-5
L2SP_DECODER = 1e-5
CLIP_DECODER, CLIP_OTHER = 1.0, 5.0
EFF, EPOCHS = 256, 8
MACRO_TOL = 0.0015
LAMBDA_I  = 0.25
D_Q, D_M, D_HID = 64, 8, 128
GATE_DROPOUT = 0.0

# --- BIEN THI NGHIEM DUY NHAT: ASL -> WASL alpha=.50 tren z_MM ---
WASL_ALPHA = 0.50
W_LO, W_HI = 0.5, 3.0
ARMS      = ['B1', 'B1u']
ARM_LABEL = {'B1': 'B1  gate + WASL.50 (EHR that)', 'B1u': 'B1u gate + WASL.50 (NULL EHR)'}
ARM_COND  = {'B1': True, 'B1u': False}
WORST3_FIXED = ['Fracture', 'Pleural_Other', 'Enlarged_Cardiomediastinum']
SHUFFLE_SEED = 0

# --- moc nb12: WASL .50 toan cuc tren kien truc CU (adapter block 10/11) ---
NB12_WASL = dict(tail_peak=0.1230, tail_peak_ep=2, tail_ep8=0.1098, tail_drop=0.0132,
                 macro=0.4405, tail=0.1230, worst3_fixed=0.1388, tail_recall_k=0.1963)

PREREG = dict(
    experiment='Stage B (rut gon) — disease gate + global WASL alpha=.50',
    deviation=('Stage A KHONG dat cong prereg (G_DA_macro +0.0011, G_DA_R3 +0.0007 tai ckpt chon). '
               'Chay tiep dua tren: (1) REAL-SHUFFLE +0.0028 > REAL-NULL +0.0012 -> SHUFFLE te hon NULL '
               '=> dieu kien hoa theo NOI DUNG; (2) bootstrap R3 +0.0007 [+0.0001,+0.0013] loai tru 0; '
               '(3) G_DA_R3 theo epoch dat +0.0023@ep5 nhung §11 boc ep2. '
               'GHI RO: 1 seed, chech prereg, hieu ung gate nho hon nguong thuc dung.'),
    only_variable='loss tren z_MM: ASL -> WASL alpha=.50 (positive term only)',
    arms=ARMS, arm_cond=ARM_COND,
    baseline='B0 KHONG train lai — dung A1 cua nb14 (trung khit cau hinh DA-E + ASL)',
    loss=f'WASL(alpha={WASL_ALPHA})(z_MM) + {LAMBDA_I}*ASL(z_IMG)  — L_IMG giu ASL khong trong so',
    why_not_B2B3=('B2/B3 dinh tuyen phan nhan them CHI qua nhanh residual da co lap; Stage A do duoc '
                  'nhanh do chi dich R3 ~+0.0007 (CI <= +0.0013) -> tran bi chan. Hoan.'),
    seed=SEED, epochs=EPOCHS, early_stopping=False,
    # --- BAT BUOC: cac cell tai dung verbatim tu nb14 doc PREREG['d_e'|'wd'|'warmup_frac'] ---
    d_e=128, wd=1e-2, warmup_frac=0.05,
    lr_adapter=LR_ADAPTER, lr_decoder=LR_DECODER, l2sp=L2SP_DECODER,
    decoder_warmup_epochs=DECODER_WARMUP_EPOCHS, effective_batch=EFF,
    estimand=['Delta_WASL = B1 - A1', 'G_DA|WASL = B1 - B1u', 'G_DA|ASL = A1 - A0 (nb14)',
              'I_loss x EHR = (B1-B1u) - (A1-A0)'],
    stability='so peak / epoch of peak / ep8 / peak-drop voi WASL .50 toan cuc cua nb12 (drop 0.0132)',
    ckpt_rule=f'macro >= best-{MACRO_TOL} -> max tail (tie: worst3_fixed, recall@K, macro)',
    go_no_go=('vs A1: dRare3 >= +0.002, dMacro >= -0.002, dNonTarget10 >= -0.0015, '
              '>=2/3 nhan Rare-3 cai thien'),
    eval='VAL only, study-level pool 7:3, NO TEST')
json.dump(PREREG, open(f'{SC_DIR}/preregistration.json','w'), indent=1)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '|', torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU')
if dev=='cuda': print(f'  VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')
for _k in ('d_e', 'wd', 'warmup_frac'):
    assert _k in PREREG, f'PREREG thieu {_k} — cell tai dung tu nb14 se KeyError'
print(f'arms {ARMS} (B0 = A1 cua nb14, khong train lai) | WASL alpha {WASL_ALPHA} | {EPOCHS} epoch')
print('PREREG khoa ->', f'{SC_DIR}/preregistration.json')

## C0-bis · Stage ảnh (Colab). No-op nếu ảnh đã ở `LOCAL_IMG`.

In [ ]:
os.makedirs(LOCAL_IMG, exist_ok=True)
_have = len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')])
if _have > 1000 and not os.path.isdir(SHARD_DIR):
    print(f'{_have:,} anh da co san o {LOCAL_IMG}; khong co SHARD_DIR -> bo qua staging')
else:
    tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar')) if os.path.isdir(SHARD_DIR) else []
    assert tars or _have > 1000, f'khong co shard o {SHARD_DIR} va {LOCAL_IMG} rong -> can data'
    todo = [f for f in tars if not os.path.exists(os.path.join(LOCAL_IMG, f'.done_{f}'))]
    print(f'shards: {len(tars)} | to stage: {len(todo)}')
    for f in tqdm(todo, desc='staging shards', unit='shard'):
        tmp = f'/content/{f}'
        shutil.copyfile(os.path.join(SHARD_DIR, f), tmp)
        with tarfile.open(tmp) as t: t.extractall(LOCAL_IMG)
        os.remove(tmp); open(os.path.join(LOCAL_IMG, f'.done_{f}'), 'w').close()
    print('local images:', len([f for f in os.listdir(LOCAL_IMG) if f.endswith('.jpg')]))

## C1 · Cohort + Full-EHR (verbatim nb11·C1) + **`m` 8 chiều** + khoá Rare-3 / NonTarget-10

`m` là **tường minh hoá** độ tin cậy EHR (mask đã nằm sẵn trong `e`), không phải thông tin
lâm sàng mới. Tám thành phần theo `FINAL_DECISIONS` §Q7, mọi thành phần trong `[0,1]`.

In [ ]:
meta   = json.load(open(os.path.join(OUT_DIR, 'cohort_meta.json')))
LABELS = meta['labels']; assert len(LABELS) == 13
ehr = pd.read_parquet(os.path.join(OUT_DIR, 'cohort_study_level.parquet')).sort_values('study_id').reset_index(drop=True)
N = len(ehr)
Y_STUDY  = ehr[LABELS].to_numpy('float32')
EHR_ROW  = {int(s): i for i, s in enumerate(ehr.study_id.to_numpy())}

# ---- vocab + full-EHR arrays (verbatim 06b/11) ----
vocab = pickle.load(open(os.path.join(OUT_DIR, 'ehr_vocab.pkl'), 'rb'))
ORG_V, SPEC_V = vocab['org_vocab_size'], vocab['spec_vocab_size']
MISSING = vocab['reserved']['MISSING']

def as_id_lists(col):
    out = []
    for v in ehr[col].to_numpy():
        a = np.asarray(v, np.int64).ravel() if v is not None and np.ndim(v) else np.array([], np.int64)
        out.append(a if a.size else np.array([MISSING], np.int64))
    return out
RAGGED = {'t1_org':('t1_org_ids',ORG_V), 't1_spec':('t1_spec_ids',SPEC_V),
          't2_spec':('t2_spec_ids',SPEC_V), 't3_org':('t3_org_ids',ORG_V)}
bags = {k: as_id_lists(c) for k,(c,_) in RAGGED.items()}

def codes(series, cats):
    m = {c.lower(): i for i, c in enumerate(cats)}
    return series.astype(str).str.lower().map(lambda x: m.get(x, 0)).to_numpy('int64'), len(cats)
set_id,  _ = codes(ehr['setting'],        ['inpatient','ED','outpatient'])
rec_id,  _ = codes(ehr['t3_recency_bin'], ['none','7-30d','30-90d'])
stat_id    = ehr['t1_status'].to_numpy('int64')

BIN = ['t2_pending','t2_time_imprecise','t1_same_encounter']
binv = {b: ehr[b].to_numpy('float32') for b in BIN}
tr_mask = (ehr.split == 'train').to_numpy()
def zscore_train(x, use_log=True):
    z = np.log1p(np.clip(x.astype('float32'), 0, None)) if use_log else x.astype('float32')
    mu, sd = z[tr_mask].mean(), z[tr_mask].std() or 1.0
    return ((z - mu)/sd).astype('float32')
COUNTS = ['t2_n_pending','t3_n_specimens','t4_n_prior_positive_ever',
          't4_years_since_first_culture','t4_n_prior_encounters']
cnts = {c: zscore_train(ehr[c].to_numpy()) for c in COUNTS}

_z = lambda k: np.zeros((N, k), 'float32')
LABS   = vocab['labs']
lab_val  = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in LABS], 1)
lab_mask = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in LABS], 1)
VITALS = [c for c in vocab.get('vitals', []) if f'{c}_value' in ehr.columns]
vit_val  = np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in VITALS], 1) if VITALS else _z(0)
vit_mask = np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in VITALS], 1) if VITALS else _z(0)
COMO_COLS = [c for c in (vocab.get('comorbidity', []) + ['como_mask']) if c in ehr.columns]
como_arr  = np.stack([ehr[c].to_numpy('float32') for c in COMO_COLS], 1) if COMO_COLS else _z(0)
REC1 = ['t1_days_since_last_specimen','t1_days_since_last_positive']; REC3 = ['t3_days_since_last_positive']
rec_block = lambda ns: (np.stack([ehr[f'{c}_value'].to_numpy('float32') for c in ns], 1),
                        np.stack([ehr[f'{c}_mask'].to_numpy('float32')  for c in ns], 1))
rec1_v, rec1_m = rec_block(REC1); rec3_v, rec3_m = rec_block(REC3)
MICRO_TIERS = (1, 2, 3)

cf = pickle.load(open(os.path.join(OUT_DIR, 'class_freq.pkl'), 'rb'))
N_POS_TRAIN = np.asarray(cf['class_freq'], dtype='float64')     # study-level TRAIN positives
N_TRAIN     = int(cf['num_train'])
TRAIN_PREV  = N_POS_TRAIN / N_TRAIN
TIER = np.where(TRAIN_PREV<0.02,'tail', np.where(TRAIN_PREV<0.10,'medium','head'))
TAIL_IDX = np.where(TIER=='tail')[0]
print(f'cohort {N:,} | labs {len(LABS)} | vitals {len(VITALS)} | como {COMO_COLS} | '
      f'micro org/spec {ORG_V}/{SPEC_V} | tiers {MICRO_TIERS}')
print(f'train studies {N_TRAIN:,} | tiers nhan: {dict(zip(*np.unique(TIER,return_counts=True)))}')
print('tail labels:', [LABELS[j] for j in TAIL_IDX])

vm = pd.read_parquet(os.path.join(OUT_DIR, 'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal','lateral'])]
_dup = [c for c in ('subject_id','split','setting') if c in vm.columns]
man = vm.drop(columns=_dup).merge(ehr[['study_id','subject_id','split']], on='study_id', how='inner')
before = len(man)
man = man[man.dicom_id.map(lambda d: os.path.exists(f'{LOCAL_IMG}/{d}.jpg'))].sort_values('dicom_id').reset_index(drop=True)
assert len(man), f'ALL {before:,} images missing -> chay C0-bis'
assert (before-len(man))/before < 0.01, f'{before-len(man)} images missing -> chay C0-bis'
man['ehr_row'] = man.study_id.map(EHR_ROW).astype('int64')
IDX = {s: np.where(man.split.to_numpy()==s)[0] for s in ('train','val','test')}
print('images by split:', {s: int(len(IDX[s])) for s in IDX})
print('NOTE: test index ton tai nhung KHONG BAO GIO dung trong screen nay.')

# ================= m: 8-d completeness summary (Q7 KHOA) =================
_micro_n = np.zeros(N, 'float32')
for _nm in RAGGED:
    _micro_n += np.array([int((a != MISSING).sum()) for a in bags[_nm]], 'float32')
_MICRO_MAX = float(_micro_n.max())
_como_mask = (ehr['como_mask'].to_numpy('float32') if 'como_mask' in ehr.columns
              else np.zeros(N, 'float32'))
M_COLS = ['lab_frac','vital_frac','como_mask','t1_recency_frac','t3_recency_mask',
          'no_t1_specimen','no_t3_recency','micro_token_frac']
m_arr = np.stack([
    lab_mask.mean(1),                                              # m1
    vit_mask.mean(1) if vit_mask.shape[1] else np.zeros(N,'float32'),   # m2
    _como_mask,                                                    # m3
    rec1_m.mean(1),                                                # m4
    rec3_m[:, 0],                                                  # m5
    (stat_id == 0).astype('float32'),                              # m6: khong co specimen T1
    (rec_id == 0).astype('float32'),                               # m7: recency bin = 'none'
    (np.log1p(_micro_n) / np.log1p(_MICRO_MAX)).astype('float32'), # m8
], 1).astype('float32')
D_M_CHK = m_arr.shape[1]
assert D_M_CHK == 8 and np.isfinite(m_arr).all(), 'm loi'
assert m_arr.min() >= -1e-6 and m_arr.max() <= 1 + 1e-6, f'm ngoai [0,1]: {m_arr.min()},{m_arr.max()}'
# NULL = chu ky ALL-MISSING (khong phai all-zero ngay tho: m6/m7 phai = 1)
NULL_M_VEC = np.array([0., 0., 0., 0., 0., 1., 1., 0.], 'float32')
print('m 8-d | mean tren TRAIN:')
for _c, _v in zip(M_COLS, m_arr[tr_mask].mean(0)): print(f'   {_c:<18} {_v:.4f}')
print('   NULL signature   ', dict(zip(M_COLS, NULL_M_VEC.tolist())))

# ================= Rare-3 (KHOA tu TRAIN counts) + NonTarget-10 =================
RARE3_IDX  = np.sort(np.argsort(N_POS_TRAIN, kind='stable')[:3])
RARE3      = [LABELS[j] for j in RARE3_IDX]
NONTGT_IDX = np.setdiff1d(np.arange(len(LABELS)), RARE3_IDX)
assert set(RARE3) == {'Pleural_Other', 'Fracture', 'Lung_Lesion'}, f'Rare-3 lech: {RARE3}'
assert all(l in LABELS for l in WORST3_FIXED)
W3_IDX = np.array([LABELS.index(l) for l in WORST3_FIXED])
EC_IDX = LABELS.index('Enlarged_Cardiomediastinum')
print(f'\nRare-3 (KHOA): {[(l, int(N_POS_TRAIN[j])) for l, j in zip(RARE3, RARE3_IDX)]}')
print(f'NonTarget-10 : {len(NONTGT_IDX)} nhan con lai (safeguard chinh chong collateral damage)')
print(f'Worst-3 fixed: {WORST3_FIXED} | overlap voi Rare-3 = '
      f'{sorted(set(RARE3) & set(WORST3_FIXED))} -> "Worst-3 tang" KHONG duoc goi la '
      f'generalization ngoai muc tieu; bao cao rieng Enlarged_Cardiomediastinum (idx {EC_IDX}).')

## C2 · Dataset (verbatim nb11·C2) + `m` đi kèm cùng permutation

`m` được index bằng **cùng biến `e`** như mọi feature EHR khác ⇒ SHUFFLE tự động hoán vị
`e` và `m` theo **cùng một permutation** (yêu cầu của `FINAL_DECISIONS` §SHUFFLE).

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T
from PIL import Image
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
train_tf = T.Compose([T.RandomResizedCrop(MODEL_SIZE, scale=(0.85,1.0), ratio=(0.95,1.05),
                                          interpolation=T.InterpolationMode.BICUBIC),
                      T.RandomRotation(7), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
eval_tf  = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])

class ImgDS(Dataset):
    def __init__(self, idx, tf, perm=None): self.idx, self.tf, self.perm = np.asarray(idx), tf, perm
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        r = man.iloc[self.idx[k]]
        im = Image.open(f'{LOCAL_IMG}/{r.dicom_id}.jpg').convert('L')
        im = Image.merge('RGB', (im, im, im))
        tr = int(r.ehr_row)
        e  = tr if self.perm is None else int(self.perm[tr])
        d = dict(image=self.tf(im), y=torch.from_numpy(Y_STUDY[tr]),
                 set_id=set_id[e], rec_id=rec_id[e], stat_id=stat_id[e],
                 lab_val=torch.from_numpy(lab_val[e]), lab_mask=torch.from_numpy(lab_mask[e]),
                 vit_val=torch.from_numpy(vit_val[e]), vit_mask=torch.from_numpy(vit_mask[e]),
                 como=torch.from_numpy(como_arr[e]),
                 rec1_v=torch.from_numpy(rec1_v[e]), rec1_m=torch.from_numpy(rec1_m[e]),
                 rec3_v=torch.from_numpy(rec3_v[e]), rec3_m=torch.from_numpy(rec3_m[e]),
                 m=torch.from_numpy(m_arr[e]))
        for _bn in BIN:    d[_bn] = binv[_bn][e]
        for _c in COUNTS:  d[_c]  = cnts[_c][e]
        for nm in RAGGED:  d[nm]  = torch.from_numpy(bags[nm][e])
        return d

def collate(bb):
    o = dict(image=torch.stack([x['image'] for x in bb]), y=torch.stack([x['y'] for x in bb]),
             set_id=torch.tensor([x['set_id'] for x in bb]),
             rec_id=torch.tensor([x['rec_id'] for x in bb]),
             stat_id=torch.tensor([x['stat_id'] for x in bb]))
    for kk in ('lab_val','lab_mask','vit_val','vit_mask','como','rec1_v','rec1_m','rec3_v','rec3_m','m'):
        o[kk] = torch.stack([x[kk] for x in bb])
    for kk in BIN + COUNTS: o[kk] = torch.tensor([x[kk] for x in bb], dtype=torch.float32)
    for nm in RAGGED:
        lens = [len(x[nm]) for x in bb]
        o[nm] = (torch.cat([x[nm] for x in bb]) if sum(lens) else torch.empty(0, dtype=torch.int64),
                 torch.tensor([0]+lens[:-1]).cumsum(0))
    return o

def loader(split, tf, shuffle, bs, perm=None, workers=8):
    return DataLoader(ImgDS(IDX[split], tf, perm), batch_size=bs, shuffle=shuffle, collate_fn=collate,
                      num_workers=workers, pin_memory=(dev=='cuda'), persistent_workers=(workers>0))
to_dev = lambda b: {k:(tuple(t.to(dev, non_blocking=True) for t in v) if isinstance(v, tuple)
                       else v.to(dev, non_blocking=True)) for k, v in b.items()}
def shuffle_perm(seed=SHUFFLE_SEED):
    rng = np.random.default_rng(seed); p = np.arange(N)
    for s in ('train','val','test'):                       # trao TRONG tung split
        rows = np.unique(man.iloc[IDX[s]].ehr_row.to_numpy()); pr = rows.copy(); rng.shuffle(pr); p[rows] = pr
    return p
print('loaders ready (full-EHR + m 8-d, cung permutation)')

## C3 · Backbone đông cứng — **toàn bộ 12 block `no_grad`**

Không còn adapter ⇒ `forward_img_tokens` chạy hết backbone + `layernorm` một lần mỗi batch,
dùng chung cho cả hai arm. Rẻ hơn hẳn nb11–13 (vốn backprop qua 3 block cuối).

In [ ]:
from transformers import AutoModel
backbone = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
assert os.path.exists(BB_CKPT), f'04b backbone khong thay: {BB_CKPT}'
_sd = torch.load(BB_CKPT, map_location=dev, weights_only=False)['model']
_bb = {k[9:]: v for k, v in _sd.items() if k.startswith('backbone.')}
assert _bb, 'no backbone.* keys in 04b checkpoint'
backbone.load_state_dict(_bb, strict=True)
for p in backbone.parameters(): p.requires_grad = False
LAYERS = backbone.encoder.layer; n_blocks = len(LAYERS); D = backbone.config.hidden_size
assert D == 768
print(f'blocks {n_blocks} (TAT CA no_grad) | hidden {D}')

def _layer(h, i):
    o = LAYERS[i](h); return o[0] if isinstance(o, (tuple, list)) else o

@torch.no_grad()
def forward_img_tokens(x):
    """RAD-DINO -> patch token da layernorm. Dung chung moi arm, khong gradient."""
    h = backbone.embeddings(x)
    for i in range(n_blocks): h = _layer(h, i)
    return backbone.layernorm(h).detach()           # [B, 1+Npatch, 768]

# G0: manual loop == stock forward
_x = torch.randn(2, 3, 518, 518, device=dev) if False else None
from transformers import AutoImageProcessor as _AIP
_ms = _AIP.from_pretrained('microsoft/rad-dino').crop_size['height']
_x = torch.randn(2, 3, _ms, _ms, device=dev)
with torch.no_grad():
    ref = backbone(pixel_values=_x).last_hidden_state
    got = forward_img_tokens(_x)
_d = (ref - got).abs().max().item()
assert torch.allclose(ref, got, atol=1e-4), f'manual != stock (max {_d:.2e})'
print(f'G0 forward_img_tokens == backbone(x).last_hidden_state  max|diff| {_d:.2e}  OK '
      f'| tokens {got.shape[1]} (1 CLS + {got.shape[1]-1} patch)')

## C4 · Full EhrEncoder (verbatim 06b/nb11)

`CondAdapter` **không** còn dùng (bỏ theo quyết định Q1) nên không copy sang.

In [ ]:
Act = nn.GELU

class EhrEncoder(nn.Module):
    def __init__(self, d_e=PREREG['d_e'], d_hidden=256, dropout=0.2):
        super().__init__(); T = set(MICRO_TIERS); self.T = T
        self.mods = nn.ModuleDict(); d = 0
        def bag(key, vsize, dim):
            self.mods[key] = nn.EmbeddingBag(vsize, dim, mode='sum'); return dim
        self.set_emb = nn.Embedding(3, 4); d += 4
        d += 2*len(LABS) + 2*len(VITALS) + len(COMO_COLS)
        self.n_vit, self.n_como = len(VITALS), len(COMO_COLS)
        if 1 in T:
            d += bag('t1_org', ORG_V, 16) + bag('t1_spec', SPEC_V, 8)
            self.stat_emb = nn.Embedding(4, 4); d += 4 + 1 + 2*len(REC1)
        if 2 in T: d += bag('t2_spec', SPEC_V, 8) + 3
        if 3 in T:
            d += bag('t3_org', ORG_V, 16); self.rec_emb = nn.Embedding(3, 4)
            d += 4 + 2*len(REC3) + 1
        self.in_dim = d
        self.mlp = nn.Sequential(nn.Linear(d, d_hidden), nn.LayerNorm(d_hidden), Act(),
                                 nn.Dropout(dropout),
                                 nn.Linear(d_hidden, d_e), nn.LayerNorm(d_e), Act())
    def _bag(self, k, b):
        flat, off = b[k]; m = self.mods[k]
        return m(flat, off)
    def forward(self, b):
        T = self.T
        p = [self.set_emb(b['set_id']), b['lab_val'], b['lab_mask']]
        if self.n_vit:  p += [b['vit_val'], b['vit_mask']]
        if self.n_como: p += [b['como']]
        if 1 in T: p += [self._bag('t1_org',b), self._bag('t1_spec',b), self.stat_emb(b['stat_id']),
                         b['t1_same_encounter'][:,None], b['rec1_v'], b['rec1_m']]
        if 2 in T: p += [self._bag('t2_spec',b), b['t2_pending'][:,None],
                         b['t2_time_imprecise'][:,None], b['t2_n_pending'][:,None]]
        if 3 in T: p += [self._bag('t3_org',b), self.rec_emb(b['rec_id']), b['rec3_v'], b['rec3_m'],
                         b['t3_n_specimens'][:,None]]
        return self.mlp(torch.cat(p, 1))

print('EhrEncoder in_dim:', EhrEncoder().in_dim, '-> d_e', PREREG['d_e'],
      '| params:', sum(p.numel() for p in EhrEncoder().parameters()))
print('CondAdapter: KHONG dung o Stage A (bo theo quyet dinh Q1)')

## C5 · `ImageHead` (hook lấy `h^IMG`) + `GateResidual` + `DiseaseGateModel`

`h_c^IMG` lấy bằng **forward-hook** trên `mld.decoder` thay vì viết lại thân `MLDecoder.forward`
— an toàn hơn, không phụ thuộc chi tiết source. Output của `nn.TransformerDecoder` là
`[Q, B, 768]`; với `num_of_groups = num_classes = 13` thì `Q = 13`, `duplicate_factor = 1`
⇒ ánh xạ 1:1 class ↔ query. C6 assert lại điều này lúc chạy.

`isolate=True` cho **auxiliary forward** của Stage B: `sg(z_IMG)`, `sg(e)`, `sg(m)`, **cùng
bộ tham số**. Stage A không dùng, nhưng hiện thực sẵn và test ngay để de-risk Stage B.

In [ ]:
import sys, types
ML_SRC = '/content/ml_decoder.py'
if not os.path.exists(ML_SRC):
    !wget -q -O {ML_SRC} https://raw.githubusercontent.com/Alibaba-MIIL/ML_Decoder/main/src_files/ml_decoder/ml_decoder.py
sys.path.insert(0, '/content')
import ml_decoder as _mld
from ml_decoder import MLDecoder
if not hasattr(_mld.TransformerDecoderLayerOptimal, '_patched'):
    _oi = _mld.TransformerDecoderLayerOptimal.__init__
    def _init(self, *a, **kw): _oi(self, *a, **kw); self.self_attn = types.SimpleNamespace(batch_first=False)
    _mld.TransformerDecoderLayerOptimal.__init__ = _init
    _of = _mld.TransformerDecoderLayerOptimal.forward
    def _fwd(self, tgt, memory, **kw):
        kw.pop('tgt_is_causal', None); kw.pop('memory_is_causal', None); return _of(self, tgt, memory, **kw)
    _mld.TransformerDecoderLayerOptimal.forward = _fwd
    _mld.TransformerDecoderLayerOptimal._patched = True

assert os.path.exists(H5_CKPT), f'H5 head khong thay: {H5_CKPT}'
H5_HEAD = torch.load(H5_CKPT, map_location=dev, weights_only=False)['model']
def _sub(pfx): return {k[len(pfx):]: v for k, v in H5_HEAD.items() if k.startswith(pfx)}
assert _sub('norm.') and _sub('mld.'), f'H5 thieu norm.*/mld.*: {list(H5_HEAD)[:5]}'
print('H5 head <- token_fusion/checkpoints/H5_s42_best.pt | norm', len(_sub('norm.')), 'mld', len(_sub('mld.')))

class ImageHead(nn.Module):
    """LayerNorm + MLDecoder (warm-start H5). Tra ve (z_IMG [B,13], h_IMG [B,13,768])."""
    def __init__(s):
        super().__init__()
        s.norm = nn.LayerNorm(768)
        s.mld  = MLDecoder(num_classes=len(LABELS), initial_num_features=768,
                           decoder_embedding=768, num_of_groups=len(LABELS))
        s.norm.load_state_dict(_sub('norm.')); s.mld.load_state_dict(_sub('mld.'))
        s._cache = {}
        s.mld.decoder.register_forward_hook(lambda mod, inp, out: s._cache.__setitem__('h', out))
    def forward(s, tok):
        z = s.mld(s.norm(tok))                       # [B, 13]
        h = s._cache.pop('h')                        # [Q, B, 768] (batch_first=False)
        return z, h.transpose(0, 1)                  # [B, 13, 768]

class GateResidual(nn.Module):
    """g_c = sigmoid(f_g([sg(h_c), q_c, e, m])) ; delta_c = f_delta([q_c, e, m]).

    MLP dung chung cho ca 13 benh (xem caveat Q5: encoder-isolated, KHONG phai
    class-parameter-isolated) -> NonTarget-10 la bat buoc o Stage B.
    """
    def __init__(s, d_q=D_Q, d_m=D_M, d_e=PREREG['d_e'], d_h=768, hid=D_HID, p=GATE_DROPOUT):
        super().__init__()
        s.q = nn.Embedding(len(LABELS), d_q); nn.init.normal_(s.q.weight, std=0.02)
        mk = lambda din: nn.Sequential(nn.Linear(din, hid), nn.GELU(), nn.Dropout(p), nn.Linear(hid, 1))
        s.fg = mk(d_h + d_q + d_e + d_m)
        s.fd = mk(d_q + d_e + d_m)
        nn.init.zeros_(s.fg[-1].bias)                                  # gate bias = 0 -> g ~ .5
        nn.init.zeros_(s.fd[-1].weight); nn.init.zeros_(s.fd[-1].bias) # delta ZERO-INIT
    def forward(s, h_img, e, m, isolate=False):
        B, C = h_img.shape[0], len(LABELS)
        hd = h_img.detach()                                            # sg(h_IMG) LUON LUON
        if isolate: e, m = e.detach(), m.detach()                      # sg(e), sg(m) cho aux
        q  = s.q.weight.unsqueeze(0).expand(B, -1, -1)                 # q KHONG detach (aux duoc cap nhat q)
        ee = e.unsqueeze(1).expand(-1, C, -1)
        mm = m.unsqueeze(1).expand(-1, C, -1)
        g  = torch.sigmoid(s.fg(torch.cat([hd, q, ee, mm], -1))).squeeze(-1)
        d  = s.fd(torch.cat([q, ee, mm], -1)).squeeze(-1)
        return g, d

def NULL_BATCH(B, device):
    """Batch 'moi thu missing' — hang qua moi sample (verbatim nb11)."""
    z1 = lambda w: torch.zeros(B, w, device=device)
    sc = lambda: torch.zeros(B, device=device)
    idl = lambda: torch.zeros(B, dtype=torch.long, device=device)
    d = dict(set_id=idl(), stat_id=idl(), rec_id=idl(),
             lab_val=z1(lab_val.shape[1]), lab_mask=z1(lab_mask.shape[1]),
             vit_val=z1(vit_val.shape[1]), vit_mask=z1(vit_mask.shape[1]),
             como=z1(como_arr.shape[1]),
             rec1_v=z1(rec1_v.shape[1]), rec1_m=z1(rec1_m.shape[1]),
             rec3_v=z1(rec3_v.shape[1]), rec3_m=z1(rec3_m.shape[1]))
    for kk in ('t1_same_encounter','t2_pending','t2_time_imprecise','t2_n_pending','t3_n_specimens'):
        d[kk] = sc()
    for nm in RAGGED:
        d[nm] = (torch.full((B,), MISSING, dtype=torch.long, device=device),
                 torch.arange(B, dtype=torch.long, device=device))
    return d

_NULL_M_T = torch.as_tensor(NULL_M_VEC)
def NULL_M(B, device): return _NULL_M_T.to(device).unsqueeze(0).expand(B, -1)

class DiseaseGateModel(nn.Module):
    def __init__(s, cond=True):
        super().__init__(); s.cond = cond
        s.img = ImageHead(); s.enc = EhrEncoder(); s.gr = GateResidual()
    def ehr(s, batch, B, device):
        if s.cond: return s.enc(batch), batch['m']
        return s.enc(NULL_BATCH(B, device)), NULL_M(B, device)
    def forward(s, tok, batch, isolate=False, parts=False):
        z_img, h_img = s.img(tok)
        e, m = s.ehr(batch, tok.shape[0], tok.device)
        g, d = s.gr(h_img, e, m, isolate=isolate)
        base = z_img.detach() if isolate else z_img
        z_mm = base + g * d
        if parts: return z_mm, z_img, g, d
        return z_mm, z_img

def build_model(seed=SEED, cond=True):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    return DiseaseGateModel(cond).to(dev)

def clone_model(src, cond):
    m = DiseaseGateModel(cond).to(dev)
    m.load_state_dict({k: v.detach().clone() for k, v in src.state_dict().items()})
    return m

dec_params = lambda m: list(m.img.norm.parameters()) + list(m.img.mld.parameters())
oth_params = lambda m: list(m.enc.parameters()) + list(m.gr.parameters())

D0 = ImageHead().to(dev).eval()                       # anchor H5 nguyen ban, khong train
for p in D0.parameters(): p.requires_grad_(False)
H5_DEC = [p.detach().clone() for p in (list(D0.norm.parameters()) + list(D0.mld.parameters()))]

_M0 = build_model(SEED, cond=True)
_n = lambda ps: sum(p.numel() for p in ps)
print(f'trainable: image head {_n(dec_params(_M0))/1e6:.3f}M | EHR enc {_n(list(_M0.enc.parameters()))/1e3:.1f}k'
      f' | gate+residual+q {_n(list(_M0.gr.parameters()))/1e3:.1f}k'
      f'  => tong {_n(list(_M0.parameters()))/1e6:.3f}M')
print(f'  q embedding {tuple(_M0.gr.q.weight.shape)} | snapshot ~{_n(list(_M0.parameters()))*4/1e6:.0f} MB/arm')
_c = clone_model(_M0, True)
assert all(torch.equal(a, b) for a, b in zip(_M0.state_dict().values(), _c.state_dict().values()))
print('clone_model bit-for-bit OK'); del _c

## C6 · WASL α=.50 + class weights + cổng

`WASL` dùng **đúng công thức khoá của nb12**: chỉ nhân `w_c` vào **positive term**, negative
term giữ nguyên; `w = clip(s·n_c^(−α), 0.5, 3.0)` với `s` tìm bằng bisection để `mean(w) = 1`.

Cổng bắt buộc: `WASL(w=1)` phải **bit-for-bit** bằng ASL của nb11/12/13/14, và kiến trúc phải
qua lại toàn bộ G1–G9 của nb14 (dán lại nguyên xi, vì kiến trúc không đổi).

In [ ]:
def class_weights(n_pos, alpha, lo=W_LO, hi=W_HI, tol=1e-12, iters=200):
    """w = clip(s * n^-alpha, lo, hi), s tim bang bisection de mean(w)=1. Verbatim nb12."""
    n = np.asarray(n_pos, dtype='float64'); assert np.all(n > 0)
    if alpha == 0: return np.ones(len(n))
    x = n ** (-alpha)
    f = lambda s: np.clip(s*x, lo, hi).mean() - 1.0
    s_lo, s_hi = 1e-12, 1.0
    while f(s_hi) < 0: s_hi *= 2.0
    for _ in range(iters):
        s = 0.5*(s_lo + s_hi)
        if f(s) < 0: s_lo = s
        else:        s_hi = s
        if s_hi - s_lo < tol*max(1.0, s_hi): break
    return np.clip(0.5*(s_lo+s_hi)*x, lo, hi)

W_VEC = class_weights(N_POS_TRAIN, WASL_ALPHA)
wtab = pd.DataFrame({'Label': LABELS, 'N pos': N_POS_TRAIN.astype(int), 'Tier': TIER,
                     'in Rare-3': [l in RARE3 for l in LABELS], 'w': W_VEC}
                   ).sort_values('N pos', ascending=False).reset_index(drop=True)
print(wtab.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f'\nalpha={WASL_ALPHA}: mean {W_VEC.mean():.8f} | min {W_VEC.min():.4f} | max {W_VEC.max():.4f} '
      f'| clip_lo {(W_VEC<=W_LO+1e-9).sum()} | Rare-3 w = '
      f'{ {l: round(float(W_VEC[j]),4) for l,j in zip(RARE3, RARE3_IDX)} }')
assert abs(W_VEC.mean()-1) < 1e-6 and W_VEC.min() >= W_LO-1e-9 and W_VEC.max() <= W_HI+1e-9
np.save(f'{SC_DIR}/metrics/class_weights_alpha50.npy', W_VEC)

class WASLArm(nn.Module):
    """ASL + inter-class weight tren RIENG positive term. w=1 -> identical ASL. Verbatim nb12."""
    def __init__(s, w, gn=4, gp=1, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
        s.register_buffer('w', torch.as_tensor(np.asarray(w, 'float32')))
    def forward(s, x, y, parts=False):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lp = y*torch.log(xp.clamp(min=s.eps)); ln = (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y); mod = torch.pow(1-pt, g)
        Lp, Ln = -(s.w*lp)*mod, -ln*mod
        loss = (Lp+Ln).mean()
        if parts: return loss, Lp.mean(0).detach(), Ln.mean(0).detach()
        return loss

def _asl_ref(x, y, gn=4, gp=1, clip=0.05, eps=1e-8):
    y=y.float(); xp=torch.sigmoid(x); xn=1-xp
    if clip: xn=(xn+clip).clamp(max=1)
    los=y*torch.log(xp.clamp(min=eps))+(1-y)*torch.log(xn.clamp(min=eps))
    pt=xp*y+xn*(1-y); g=gp*y+gn*(1-y); return -(los*torch.pow(1-pt,g)).mean()

torch.manual_seed(0)
_x = (torch.randn(96, len(LABELS), device=dev)*3).requires_grad_(True)
_y = (torch.rand(96, len(LABELS), device=dev) < .12).float()
_one = WASLArm(np.ones(len(LABELS))).to(dev)
print(f'\nW1 WASL(w=1) {_one(_x,_y).item():.10f} vs ASL {_asl_ref(_x,_y).item():.10f} '
      f'| equal={torch.equal(_one(_x,_y), _asl_ref(_x,_y))}')
assert torch.equal(_one(_x, _y), _asl_ref(_x, _y)), 'WASL(w=1) != ASL nb11/12/13/14'
_g1 = torch.autograd.grad(_one(_x, _y), _x, retain_graph=True)[0]
_x2 = _x.detach().clone().requires_grad_(True)
_g2 = torch.autograd.grad(_asl_ref(_x2, _y), _x2)[0]
assert torch.equal(_g1, _g2); print('   gradient cung identical  OK')

_w50 = WASLArm(W_VEC).to(dev)
_, _p1, _n1 = _one(_x, _y, parts=True)
_, _p2, _n2 = _w50(_x, _y, parts=True)
_j = int(RARE3_IDX[np.argmin(N_POS_TRAIN[RARE3_IDX])])
print(f'W2 "{LABELS[_j]}" w={W_VEC[_j]:.4f}: pos {_p1[_j]:.6f} -> {_p2[_j]:.6f} '
      f'(x{(_p2[_j]/_p1[_j]).item():.4f}) | max|dneg| toan bo {(_n2-_n1).abs().max().item():.2e} (=0)')
assert torch.equal(_n1, _n2) and abs((_p2[_j]/_p1[_j]).item() - W_VEC[_j]) < 1e-4
print('   negative term KHONG doi  OK')
print('\nCONG LOSS OK — chay tiep cell G1-G9 (kien truc) ben duoi truoc khi train')

## C6 · Cổng kiến trúc (bắt buộc pass trước khi chạy GPU)

| # | kiểm tra |
|---|---|
| G1 | `h^IMG` shape `[B,13,768]`, `duplicate_factor == 1`, 1:1 class ↔ query |
| G2 | `z_IMG` **bằng đúng** anchor H5 (`D0`) — warm-start không lệch |
| G3 | **step-0 identity**: `max\|z_MM − z_IMG\| < 1e-6` (eval FP32) — do `δ` zero-init |
| G4 | `g ≈ 0.5` lúc init (gate bias = 0) |
| G5 | **aux forward**: `max\|z_AUX − z_MM\| < 1e-6` về **giá trị**, nhưng gradient khác — de-risk Stage B |
| G6 | EHR đọc được labs + micro; `m` thay đổi ⇒ gate thay đổi |
| G7 | `NULL` = chữ ký all-missing, hằng qua sample, khác `REAL` |
| G8 | L2-SP = 0 tại init, > 0 khi image head trôi |

In [ ]:
_b = next(iter(loader('val', eval_tf, False, 4, workers=0))); _b = to_dev(_b)
_tok = forward_img_tokens(_b['image'])
_M0.eval(); D0.eval()

with torch.no_grad():
    _z_img, _h_img = _M0.img(_tok)
    # ML-Decoder gan duplicate_factor / num_classes len chinh self.decoder (khong phai self).
    # Tra ca hai cho chac, phong khi ban ml_decoder.py khac.
    _pick = lambda n: next((getattr(o, n) for o in (_M0.img.mld.decoder, _M0.img.mld)
                            if hasattr(o, n)), None)
    _dupf, _ncls = _pick('duplicate_factor'), _pick('num_classes')
    print(f'G1 h_IMG {tuple(_h_img.shape)} | duplicate_factor {_dupf} | num_classes {_ncls}')
    assert _h_img.shape == (_tok.shape[0], len(LABELS), 768), 'h_IMG sai shape -> 1:1 class<->query bi pha'
    assert _dupf == 1, f'duplicate_factor = {_dupf} != 1 -> query khong anh xa 1:1 vao class'
    assert _ncls == len(LABELS), f'num_classes = {_ncls} != {len(LABELS)}'
    print('   OK 1:1 class <-> query')

    _z0 = D0(_tok)[0]
    _d = (_z_img - _z0).abs().max().item()
    print(f'G2 |z_IMG - D0(H5)| = {_d:.2e}', 'OK' if _d < 1e-5 else '!! warm-start lech'); assert _d < 1e-5

    _z_mm, _z_i, _g, _dl = _M0(_tok, _b, parts=True)
    _d = (_z_mm - _z_i).abs().max().item()
    print(f'G3 step-0 |z_MM - z_IMG| = {_d:.2e} (<1e-6)', 'OK' if _d < 1e-6 else '!! KHONG identity')
    assert _d < 1e-6 and _dl.abs().max().item() == 0.0, 'delta phai = 0 tuyet doi luc init'

    print(f'G4 gate init: mean {_g.mean().item():.4f} | p05 {_g.flatten().quantile(.05).item():.4f} '
          f'| p95 {_g.flatten().quantile(.95).item():.4f}  (ky vong ~0.5)')
    assert 0.3 < _g.mean().item() < 0.7

    _z_aux, _, _, _ = _M0(_tok, _b, isolate=True, parts=True)
    _d = (_z_aux - _z_mm).abs().max().item()
    print(f'G5 aux forward |z_AUX - z_MM| = {_d:.2e} (<1e-6, dropout={GATE_DROPOUT})',
          'OK' if _d < 1e-6 else '!! khac gia tri')
    assert _d < 1e-6

    _e1 = _M0.enc(_b)
    _bl = dict(_b); _bl['lab_val'] = _b['lab_val'] + 3.0
    _d_lab = (_e1 - _M0.enc(_bl)).abs().mean().item()
    _cat, _off = _b['t1_org']; _bm = dict(_b)
    _bm['t1_org'] = (torch.full_like(_cat, (int(_cat.max())+1) % ORG_V), _off)
    _d_mic = (_e1 - _M0.enc(_bm)).abs().mean().item()
    _bmm = dict(_b); _bmm['m'] = 1.0 - _b['m']
    _g2 = _M0(_tok, _bmm, parts=True)[2]
    _d_m = (_g - _g2).abs().mean().item()
    print(f'G6 |de| labs {_d_lab:.3f} | micro {_d_mic:.4f} | m doi -> |dg| {_d_m:.4f} (deu > 0)')
    assert _d_lab > 1e-3 and _d_mic > 1e-4 and _d_m > 1e-5

    _en = _M0.enc(NULL_BATCH(4, dev)); _mn = NULL_M(4, dev)
    print(f'G7 NULL: std giua sample {_en.std(0).max().item():.2e} (~0) | |real-null| '
          f'{(_e1-_en).abs().mean().item():.3f} (>0) | m_NULL {_mn[0].tolist()}')
    assert _en.std(0).max().item() < 1e-4 and (_e1-_en).abs().mean().item() > 1e-3
    assert _mn[0, 5].item() == 1.0 and _mn[0, 6].item() == 1.0, 'NULL m phai la chu ky all-missing'

l2sp_term = lambda m: sum(((p-r)**2).sum() for p, r in zip(dec_params(m), H5_DEC))
_l0 = l2sp_term(_M0).item()
_p = next(iter(_M0.img.norm.parameters())); _o = _p.detach().clone()
with torch.no_grad(): _p.add_(0.01)
_l1 = l2sp_term(_M0).item()
with torch.no_grad(): _p.copy_(_o)
print(f'G8 L2-SP init {_l0:.2e} (=0) | sau khi nhich {_l1:.2e} (>0)')
assert _l0 < 1e-8 and _l1 > 0

# gradient routing cua aux (de-risk Stage B): DBL/WASL aux KHONG duoc cham enc & image head
_M0.zero_grad(set_to_none=True)
_za, _, _, _ = _M0(_tok, _b, isolate=True, parts=True)
_za[:, RARE3_IDX].sum().backward()
_ge = sum(p.grad.abs().sum().item() for p in _M0.enc.parameters() if p.grad is not None)
_gd = sum(p.grad.abs().sum().item() for p in dec_params(_M0) if p.grad is not None)
_gg = sum(p.grad.abs().sum().item() for p in _M0.gr.parameters() if p.grad is not None)
print(f'G9 aux backward: grad EHR-enc {_ge:.2e} (=0) | image head {_gd:.2e} (=0) | gate/residual {_gg:.2e} (>0)')
assert _ge == 0.0 and _gd == 0.0 and _gg > 0
_M0.zero_grad(set_to_none=True)
print('\nGATES KIEN TRUC OK')

## C7 · Predict + pool 7:3 + bộ metric + gate diagnostics + paired bootstrap

Panel khoá (7 metric) **+** `Rare-3 mAP`, `NonTarget-10 mAP`, `Enlarged_Cardiomediastinum AP`.
`predict` trả luôn thống kê gate trên val (rẻ, cùng một lượt forward).

In [ ]:
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score, auc

@torch.no_grad()
def predict(models, split, bs, perm=None, force_null=False):
    """Tra (P_img_level, GATE_STATS). force_null: eval duoi NULL EHR ma khong doi model."""
    for m in models.values(): m.eval()
    P = {t: np.zeros((len(IDX[split]), len(LABELS)), 'float32') for t in models}
    GS = {t: dict(g=[], gd=[], zi=[], g_lab=[], gd_lab=[]) for t in models}
    pos = 0
    for b in tqdm(loader(split, eval_tf, False, bs, perm, workers=0), desc=f'pred[{split}]', leave=False):
        b = to_dev(b); tok = forward_img_tokens(b['image']); k = tok.shape[0]
        for t, m in models.items():
            _c = m.cond
            if force_null: m.cond = False
            z_mm, z_img, g, d = m(tok, b, parts=True)
            m.cond = _c
            P[t][pos:pos+k] = torch.sigmoid(z_mm).float().cpu().numpy()
            gd = (g*d).abs()
            GS[t]['g'].append(g.float().cpu().numpy()); GS[t]['gd'].append(gd.float().cpu().numpy())
            GS[t]['zi'].append(z_img.abs().float().cpu().numpy())
        pos += k
    for t in GS:
        G = np.concatenate(GS[t]['g']); GD = np.concatenate(GS[t]['gd']); ZI = np.concatenate(GS[t]['zi'])
        GS[t] = dict(gate_mean=float(G.mean()), gate_p05=float(np.percentile(G,5)),
                     gate_p50=float(np.median(G)), gate_p95=float(np.percentile(G,95)),
                     res_abs_mean=float(GD.mean()), res_over_zimg=float(GD.mean()/max(ZI.mean(),1e-9)),
                     gate_per_label=G.mean(0).tolist(), res_per_label=GD.mean(0).tolist())
    return P, GS

def pool_study(p_img, split):
    sub = man.iloc[IDX[split]]; d = pd.DataFrame(p_img, columns=LABELS)
    d['study_id'] = sub.study_id.to_numpy(); d['vg'] = sub.view_group.to_numpy()
    sid = np.sort(d.study_id.unique())
    fr = d[d.vg=='frontal'].groupby('study_id')[LABELS].mean().reindex(sid)
    la = d[d.vg=='lateral'].groupby('study_id')[LABELS].mean().reindex(sid)
    hf = fr.notna().all(1).to_numpy().astype('float32'); hl = la.notna().all(1).to_numpy().astype('float32')
    den = (W_F*hf + W_L*hl)[:, None]
    return sid, (W_F*np.nan_to_num(fr.to_numpy('float32'))*hf[:,None]
                 + W_L*np.nan_to_num(la.to_numpy('float32'))*hl[:,None]) / den

def recall_at_k(y, s):
    K = int(y.sum())
    if K == 0: return np.nan
    return float(y[np.argsort(-s, kind='stable')[:K]].sum()) / K

def _ap_vec(Y, PS):
    C = Y.shape[1]
    return np.array([average_precision_score(Y[:,j], PS[:,j]) if 0 < Y[:,j].sum() < len(Y) else np.nan
                     for j in range(C)])

def eval_all(p_img, split='val'):
    sid, PS = pool_study(p_img, split)
    Y = ehr.set_index('study_id').loc[sid, LABELS].to_numpy('float32')
    C = len(LABELS)
    ap = _ap_vec(Y, PS)
    auprc = np.full(C, np.nan); auroc = np.full(C, np.nan); rk = np.full(C, np.nan)
    for j in range(C):
        if not (0 < Y[:,j].sum() < len(Y)): continue
        pr, rc, _ = precision_recall_curve(Y[:,j], PS[:,j])
        auprc[j] = auc(rc, pr); auroc[j] = roc_auc_score(Y[:,j], PS[:,j]); rk[j] = recall_at_k(Y[:,j], PS[:,j])
    T = TAIL_IDX
    return dict(macro_map=float(np.nanmean(ap)), tail_map=float(np.nanmean(ap[T])),
                rare3_map=float(np.nanmean(ap[RARE3_IDX])),
                nontarget10_map=float(np.nanmean(ap[NONTGT_IDX])),
                ec_ap=float(ap[EC_IDX]),
                worst3_fixed=float(np.nanmean(ap[W3_IDX])),
                head_map=float(np.nanmean(ap[TIER=='head'])), med_map=float(np.nanmean(ap[TIER=='medium'])),
                macro_auprc=float(np.nanmean(auprc)), macro_auroc=float(np.nanmean(auroc)),
                tail_auroc=float(np.nanmean(auroc[T])), tail_recall_k=float(np.nanmean(rk[T])),
                n_val_studies=int(len(sid)),
                per_ap=ap.tolist(), per_auprc=auprc.tolist(), per_auroc=auroc.tolist(), per_rk=rk.tolist())

METRIC_KEYS = ['macro_map','tail_map','rare3_map','nontarget10_map','worst3_fixed',
               'macro_auprc','macro_auroc','tail_auroc','tail_recall_k','ec_ap']

def paired_bootstrap(p_a, p_b, split='val', n_boot=1000, seed=0):
    """CI 95% cho delta (B - A) o muc STUDY, resample cung tap study cho ca hai arm."""
    sid, PA = pool_study(p_a, split); _, PB = pool_study(p_b, split)
    Y = ehr.set_index('study_id').loc[sid, LABELS].to_numpy('float32')
    rng = np.random.default_rng(seed); n = len(sid)
    out = {k: [] for k in ('pleural_other_ap','rare3_map','macro_map','nontarget10_map')}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        Yb = Y[idx]
        if Yb[:, RARE3_IDX].sum(0).min() == 0: continue
        aa, bb = _ap_vec(Yb, PA[idx]), _ap_vec(Yb, PB[idx])
        out['pleural_other_ap'].append(bb[TAIL_IDX[0]] - aa[TAIL_IDX[0]])
        out['rare3_map'].append(np.nanmean(bb[RARE3_IDX]) - np.nanmean(aa[RARE3_IDX]))
        out['macro_map'].append(np.nanmean(bb) - np.nanmean(aa))
        out['nontarget10_map'].append(np.nanmean(bb[NONTGT_IDX]) - np.nanmean(aa[NONTGT_IDX]))
    return {k: dict(mean=float(np.mean(v)), lo=float(np.percentile(v, 2.5)),
                    hi=float(np.percentile(v, 97.5)), n=len(v)) for k, v in out.items()}

print('predict / pool 7:3 / eval_all / paired_bootstrap ready |', len(METRIC_KEYS), 'metric')

## C8 · Train helpers + probe batch

Optimizer 2 LR-group: `{EHR encoder, gate, residual, q}` @ 1e-4 và `{image head}` @ 1e-5.
Image head đóng băng ep1, unfreeze từ ep2 kèm L2-SP về H5 (đúng protocol nb11–13).

In [ ]:
class ASLArm(nn.Module):
    """ASL cua nb11/12/13 (gp=1, gn=4, clip=.05) — bit-for-bit."""
    def __init__(s, gn=4, gp=1, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y, parts=False):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lp = y*torch.log(xp.clamp(min=s.eps)); ln = (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y); mod = torch.pow(1-pt, g)
        Lp, Ln = -lp*mod, -ln*mod
        loss = (Lp+Ln).mean()
        if parts: return loss, Lp.mean(0).detach(), Ln.mean(0).detach()
        return loss

def _asl_ref(x, y, gn=4, gp=1, clip=0.05, eps=1e-8):
    y=y.float(); xp=torch.sigmoid(x); xn=1-xp
    if clip: xn=(xn+clip).clamp(max=1)
    los=y*torch.log(xp.clamp(min=eps))+(1-y)*torch.log(xn.clamp(min=eps))
    pt=xp*y+xn*(1-y); g=gp*y+gn*(1-y); return -(los*torch.pow(1-pt,g)).mean()
_tx = torch.randn(32, len(LABELS), device=dev); _ty = (torch.rand(32, len(LABELS), device=dev) < .15).float()
assert torch.equal(ASLArm().to(dev)(_tx, _ty), _asl_ref(_tx, _ty)), 'ASL lech so voi nb11/12/13'
print('ASL bit-for-bit voi nb11/12/13  OK')

def make_opt(m):
    def split(ps, lr):
        dec=[p for p in ps if p.ndim>1]; nod=[p for p in ps if p.ndim<=1]
        return [{'params':dec,'lr':lr,'weight_decay':PREREG['wd']},
                {'params':nod,'lr':lr,'weight_decay':0.0}]
    return torch.optim.AdamW(split(oth_params(m), LR_ADAPTER) + split(dec_params(m), LR_DECODER))

def make_sched(opt, total, warm):
    cw = lambda s: s/max(warm,1) if s<warm else 0.5*(1+math.cos(math.pi*(s-warm)/max(total-warm,1)))
    const = lambda s: 1.0
    return torch.optim.lr_scheduler.LambdaLR(opt, [cw, cw, const, const])

def set_decoder_trainable(m, on):
    for p in dec_params(m): p.requires_grad_(on)

def _gnorm(ps):
    gs = [p.grad.detach() for p in ps if p.grad is not None]
    if not gs: return torch.zeros((), device=dev)
    return torch.linalg.vector_norm(torch.stack([torch.linalg.vector_norm(g) for g in gs]))

def grad_norms(m):
    return (_gnorm(dec_params(m)), _gnorm(list(m.enc.parameters())), _gnorm(list(m.gr.parameters())))

def clip_arm(m):
    d = [p for p in dec_params(m) if p.grad is not None]
    o = [p for p in oth_params(m) if p.grad is not None]
    if d: nn.utils.clip_grad_norm_(d, CLIP_DECODER)
    if o: nn.utils.clip_grad_norm_(o, CLIP_OTHER)

def sync_rng(ep, i):
    s = (SEED*1_000_003 + ep*100_003 + i) % (2**31-1)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def probe_batch(cands=(16,32,48,64,96,128)):
    if dev != 'cuda': return 8
    A = [build_model(SEED, ARM_COND[t]) for t in ARMS]; crit = ASLArm().to(dev); ok = cands[0]
    for m in A: set_decoder_trainable(m, True)
    for B in cands:
        try:
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            bb = to_dev(collate([ImgDS(IDX['train'], train_tf)[k] for k in range(B)]))
            tok = forward_img_tokens(bb['image'])
            for m in A:
                zm, zi = m(tok, bb)
                (crit(zm, bb['y']) + LAMBDA_I*crit(zi, bb['y']) + L2SP_DECODER*l2sp_term(m)).backward()
                m.zero_grad(set_to_none=True)
            pk = torch.cuda.max_memory_allocated()/1e9; tot = torch.cuda.get_device_properties(0).total_memory/1e9
            print(f'  B={B:3d} peak {pk:5.1f}/{tot:.0f} GB ({pk/tot:5.1%})')
            ok = B if pk/tot < 0.80 else ok
            if pk/tot >= 0.80: break
        except torch.cuda.OutOfMemoryError:
            print(f'  B={B} OOM'); break
    del A; torch.cuda.empty_cache(); return ok

BATCH = probe_batch(); print('-> BATCH', BATCH, '| accum', max(1, EFF//BATCH), '-> effective', EFF)

### C9-bis · Ép `BATCH` bằng đúng nb14

Nếu `BATCH` khác nb14 thì thứ tự batch khác, và `B1` mất tính paired với `A1`/`A0` — toàn bộ
estimand `Delta_WASL` và `I_loss×EHR` sụp. Cell này đọc `batch` từ resume của nb14 và ép theo.

In [ ]:
_r14 = f'{NB14_DIR}/checkpoints/_resume.pt'
if os.path.exists(_r14):
    _c14 = torch.load(_r14, map_location='cpu', weights_only=False)
    BATCH14 = int(_c14['batch']); del _c14
    print(f'BATCH cua nb14 = {BATCH14} | probe cua nb15 = {BATCH}')
    if BATCH != BATCH14:
        print(f'  -> EP BATCH = {BATCH14} de giu tinh paired voi A1/A0')
    BATCH = BATCH14
else:
    print(f'!! khong thay {_r14} — KHONG kiem tra duoc BATCH cua nb14.\n'
          f'   Neu BATCH khac nb14 thi Delta_WASL va I_loss x EHR MAT tinh paired.')
print('BATCH =', BATCH, '| accum', max(1, EFF//BATCH), '-> effective', EFF)

## C10 · Main run — `B1` và `B1u`, 8 epoch, cùng init với nb14

`build_model(SEED)` cho init trùng khít `A0`/`A1` của nb14; `torch.manual_seed(SEED*7919+ep)`
cho cùng thứ tự batch; `sync_rng(ep,i)` cho cùng dropout mask ⇒ `B1` **paired** với `A1`.

Log mỗi epoch in luôn `dWASL` (so `A1`) và `G_DA|WASL` (so `B1u`).

In [ ]:
LOG_PATH = f'{SC_DIR}/metrics/epoch_log.json'
snap_path   = lambda arm, ep: f'{SNAP_DIR}/{arm}_ep{ep}.pt'
resume_path = lambda: f'{SC_DIR}/checkpoints/_resume.pt'
NB14_LOG = json.load(open(f'{NB14_DIR}/metrics/epoch_log.json')) if \
           os.path.exists(f'{NB14_DIR}/metrics/epoch_log.json') else {}
print('nb14 log:', {k: sorted(map(int, v)) for k, v in NB14_LOG.items()} or 'KHONG THAY')

def train_all(batch=BATCH, epochs=EPOCHS):
    dl    = loader('train', train_tf, True, batch)
    accum = max(1, EFF//batch)
    TOTAL = epochs*math.ceil(len(dl)/accum); WARM = int(PREREG['warmup_frac']*TOTAL)
    crit_mm  = WASLArm(W_VEC).to(dev)          # WASL .50 tren z_MM
    crit_img = WASLArm(np.ones(len(LABELS))).to(dev)   # ASL thuan tren z_IMG (w=1 == ASL)
    _mk = lambda m: (lambda o: dict(m=m, opt=o, sch=make_sched(o, TOTAL, WARM)))(make_opt(m))

    LOG = json.load(open(LOG_PATH)) if os.path.exists(LOG_PATH) else {}
    st, start = {}, 0
    if os.path.exists(resume_path()):
        c = torch.load(resume_path(), map_location=dev, weights_only=False)
        if c['batch'] != batch: raise RuntimeError(f"resume batch {c['batch']} != {batch}")
        for t in ARMS:
            m = DiseaseGateModel(ARM_COND[t]).to(dev); m.load_state_dict(c[t]['model'])
            o = make_opt(m); o.load_state_dict(c[t]['opt'])
            s = make_sched(o, TOTAL, WARM); s.load_state_dict(c[t]['sch'])
            st[t] = dict(m=m, opt=o, sch=s)
        start = c['epoch']+1; print(f'RESUMED @ep{start+1}')
        if start >= epochs: print('da chay xong -> sang C11'); return LOG
    else:
        base = build_model(SEED, ARM_COND[ARMS[0]])
        for t in ARMS: st[t] = _mk(clone_model(base, ARM_COND[t]))
        print(f'init: {ARMS} cung init bit-for-bit voi nb14 | steps/epoch {math.ceil(len(dl)/accum)} '
              f'| total {TOTAL} | warmup {WARM}')

    for ep in range(start, epochs):
        unfrozen = ep >= DECODER_WARMUP_EPOCHS; tag = '[dec]' if unfrozen else '[warm]'
        for t in ARMS: set_decoder_trainable(st[t]['m'], unfrozen); st[t]['m'].train()
        z0 = lambda: torch.zeros(len(LABELS), device=dev)
        dg = {t: dict(l_mm=torch.zeros((), device=dev), l_img=torch.zeros((), device=dev), n=0,
                      pos=z0(), neg=z0(), gd=torch.zeros((), device=dev), ge=torch.zeros((), device=dev),
                      gg=torch.zeros((), device=dev), nopt=0) for t in ARMS}
        torch.manual_seed(SEED*7919 + ep)
        for i, b in enumerate(tqdm(dl, desc=f'ep{ep+1}/{epochs}{tag}', leave=False)):
            b = to_dev(b); tok = forward_img_tokens(b['image'])
            step_now = ((i+1) % accum == 0) or (i == len(dl)-1)
            for t in ARMS:
                e_, d_ = st[t], dg[t]
                sync_rng(ep, i)
                z_mm, z_img = e_['m'](tok, b)
                l_mm, lp, ln = crit_mm(z_mm, b['y'], parts=True)
                l_img = crit_img(z_img, b['y'])
                loss = l_mm + LAMBDA_I*l_img
                if unfrozen: loss = loss + L2SP_DECODER*l2sp_term(e_['m'])
                (loss/accum).backward()
                d_['l_mm'] += l_mm.detach(); d_['l_img'] += l_img.detach()
                d_['pos'] += lp; d_['neg'] += ln; d_['n'] += 1
                if step_now:
                    gd, ge, gg = grad_norms(e_['m'])
                    d_['gd'] += gd; d_['ge'] += ge; d_['gg'] += gg; d_['nopt'] += 1
                    clip_arm(e_['m']); e_['opt'].step(); e_['sch'].step(); e_['opt'].zero_grad(set_to_none=True)

        Pv, GS = predict({t: st[t]['m'] for t in ARMS}, 'val', batch*2)
        rows = []
        for t in ARMS:
            r = eval_all(Pv[t], 'val'); r['gate'] = GS[t]
            d_ = dg[t]; n = max(d_['n'],1); no = max(d_['nopt'],1)
            tm = lambda v: {k: float(v[TIER==k].mean()) for k in ('head','medium','tail')}
            r['diag'] = dict(loss_mm=float(d_['l_mm'].item()/n), loss_img=float(d_['l_img'].item()/n),
                             pos=tm((d_['pos']/n).cpu().numpy()), neg=tm((d_['neg']/n).cpu().numpy()),
                             grad_imghead=float(d_['gd'].item()/no), grad_enc=float(d_['ge'].item()/no),
                             grad_gate=float(d_['gg'].item()/no),
                             lr_other=st[t]['opt'].param_groups[0]['lr'], unfrozen=bool(unfrozen))
            LOG.setdefault(t, {})[str(ep+1)] = r
            torch.save({k: v.detach().cpu() for k, v in st[t]['m'].state_dict().items()}, snap_path(t, ep+1))
            rows.append(f"{t} mAP {r['macro_map']:.4f} R3 {r['rare3_map']:.4f} NT10 {r['nontarget10_map']:.4f} "
                        f"tail {r['tail_map']:.4f} g {r['gate']['gate_mean']:.3f} "
                        f"res/z {r['gate']['res_over_zimg']:.4f}")
        line = f'  ep{ep+1}{tag} | ' + ' | '.join(rows)
        _a1 = NB14_LOG.get('A1', {}).get(str(ep+1))
        if _a1:
            _b1 = LOG['B1'][str(ep+1)]
            line += (f"  || dWASL(B1-A1) mAP {_b1['macro_map']-_a1['macro_map']:+.4f} "
                     f"R3 {_b1['rare3_map']-_a1['rare3_map']:+.4f}")
        if 'B1u' in LOG and str(ep+1) in LOG['B1u']:
            _bu = LOG['B1u'][str(ep+1)]; _b1 = LOG['B1'][str(ep+1)]
            line += (f" | G_DA|WASL mAP {_b1['macro_map']-_bu['macro_map']:+.4f} "
                     f"R3 {_b1['rare3_map']-_bu['rare3_map']:+.4f}")
        json.dump(LOG, open(LOG_PATH,'w'), indent=1)
        torch.save({**{t: dict(model=st[t]['m'].state_dict(), opt=st[t]['opt'].state_dict(),
                               sch=st[t]['sch'].state_dict()) for t in ARMS},
                    'epoch': ep, 'batch': batch}, resume_path()+'.tmp')
        os.replace(resume_path()+'.tmp', resume_path())
        print(line + '  [saved]')
    print('\nDONE ->', LOG_PATH)
    return LOG

LOG = train_all()

## C11 · Chọn checkpoint · factorial · stability · go/no-go

Ba bảng:
1. **Bảng chính** `B1`/`B1u` + mốc `A1`/`A0` (nb14) tại checkpoint được chọn theo §11.
2. **Factorial** — `Delta_WASL`, `G_DA|WASL`, `G_DA|ASL`, `I_loss×EHR`.
3. **Stability** — peak / epoch of peak / cuối / sụt, so với WASL .50 toàn cục của nb12
   (kiến trúc cũ, sụt `0.0132`).

In [ ]:
LOG = json.load(open(LOG_PATH))
ALL = dict(LOG); ALL.update({k: v for k, v in NB14_LOG.items()})
LAB_ALL = dict(ARM_LABEL); LAB_ALL.update({'A1':'A1  gate + ASL (EHR that)', 'A0':'A0  gate + ASL (NULL EHR)'})

def select_ckpt(arm, src):
    eps = {int(e): src[arm][e] for e in src[arm]}
    best = max(v['macro_map'] for v in eps.values())
    cand = [e for e, v in eps.items() if v['macro_map'] >= best - MACRO_TOL]
    key = lambda e: (eps[e]['tail_map'], eps[e]['worst3_fixed'], eps[e]['tail_recall_k'], eps[e]['macro_map'])
    pick = max(cand, key=key)
    return dict(epoch=pick, metrics=eps[pick], n_cand=len(cand), best_macro=best)

ORDER = [a for a in ('B1','B1u','A1','A0') if a in ALL]
SEL = {a: select_ckpt(a, ALL) for a in ORDER}
COLS = [('macro_map','Macro mAP'), ('rare3_map','Rare-3 mAP'), ('nontarget10_map','NonTarget-10'),
        ('tail_map','Tail AP'), ('worst3_fixed','Worst-3 fixed'), ('ec_ap','EnlCardMed AP'),
        ('macro_auroc','Macro AUROC'), ('tail_recall_k','Tail Recall@K')]
main = pd.DataFrame([{'Arm': LAB_ALL[a], 'ep': SEL[a]['epoch'],
                      **{c[1]: SEL[a]['metrics'][c[0]] for c in COLS}} for a in ORDER])
print('=== BANG CHINH — VAL, study-level 7:3, seed 42 (checkpoint chon theo §11) ===')
print(main.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
main.to_csv(f'{SC_DIR}/metrics/main_table.csv', index=False)

print('\n=== FACTORIAL ===')
S = lambda a, k: SEL[a]['metrics'][k]
fac = []
for k, lab in [('macro_map','Macro'), ('rare3_map','Rare-3'), ('nontarget10_map','NonTarget-10'),
               ('tail_map','Tail AP'), ('worst3_fixed','Worst-3')]:
    row = dict(metric=lab, Delta_WASL=S('B1',k)-S('A1',k) if 'A1' in SEL else np.nan,
               G_DA_WASL=S('B1',k)-S('B1u',k),
               G_DA_ASL=S('A1',k)-S('A0',k) if 'A0' in SEL else np.nan)
    row['I_loss_x_EHR'] = row['G_DA_WASL'] - row['G_DA_ASL']
    fac.append(row)
fac = pd.DataFrame(fac)
print(fac.to_string(index=False, float_format=lambda v: f'{v:+.4f}'))
fac.to_csv(f'{SC_DIR}/metrics/factorial.csv', index=False)
print('  I_loss x EHR > 0 => EHR huu ich HON khi gradient nhan hiem du manh de fusion hoc cach khai thac.')

print('\n=== STABILITY (§9 REVISION) ===')
srows = []
for a in ORDER:
    eps = sorted(int(e) for e in ALL[a])
    for k, lab in (('rare3_map','Rare-3'), ('tail_map','Tail AP'), ('macro_map','Macro')):
        v = {e: ALL[a][str(e)][k] for e in eps}; pe = max(v, key=v.get)
        # KHONG dat ten cot la 'last'/'drop': trung ten method cua pandas ->
        # `row.drop` tra ve bound method chu khong phai gia tri -> TypeError khi format.
        srows.append(dict(arm=a, metric=lab, peak=v[pe], peak_ep=pe, final=v[max(eps)],
                          final_ep=max(eps), peak_drop=v[pe]-v[max(eps)]))
stab = pd.DataFrame(srows)
print(stab.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
stab.to_csv(f'{SC_DIR}/metrics/stability.csv', index=False)
print(f"\n  moc nb12 — WASL .50 TOAN CUC tren kien truc CU (adapter 10/11):")
print(f"    Tail AP peak {NB12_WASL['tail_peak']:.4f} @ep{NB12_WASL['tail_peak_ep']} -> "
      f"ep8 {NB12_WASL['tail_ep8']:.4f} | sut {NB12_WASL['tail_drop']:.4f} (qua 6 epoch)")
_b1t = stab[(stab['arm']=='B1') & (stab['metric']=='Tail AP')].iloc[0]
_span = int(_b1t['final_ep']) - int(_b1t['peak_ep'])
print(f"    B1 (kien truc moi, co L_IMG + L2-SP): peak {_b1t['peak']:.4f} @ep{int(_b1t['peak_ep'])} -> "
      f"ep{int(_b1t['final_ep'])} {_b1t['final']:.4f} | sut {_b1t['peak_drop']:.4f} (qua {_span} epoch)")
if _span < 6:
    print(f"    !! B1 moi co {_span} epoch sau dinh — sut nay KHONG so truc tiep duoc voi sut qua"
          f" 6 epoch cua nb12. Chay du 8 epoch roi hay ket luan ve on dinh.")
else:
    print(f"    => sut nho hon {NB12_WASL['tail_drop']-_b1t['peak_drop']:+.4f} "
          f"({'CHUA' if _b1t['peak_drop'] >= NB12_WASL['tail_drop'] else 'CO'} cai thien on dinh)")

for k, lab in (('macro_map','Macro'), ('rare3_map','Rare-3'), ('nontarget10_map','NonTarget-10')):
    t = pd.DataFrame({a: {int(e): ALL[a][e][k] for e in ALL[a]} for a in ORDER}).sort_index()
    print(f'\n--- {lab} theo epoch ---'); print(t.to_string(float_format=lambda v: f'{v:.4f}'))

### Go / no-go + control REAL/SHUFFLE/NULL tại checkpoint `B1`

In [ ]:
_m = DiseaseGateModel(True).to(dev)
_m.load_state_dict(torch.load(snap_path('B1', SEL['B1']['epoch']), map_location=dev, weights_only=False))
_m.eval()
CTRL = {}
for _mode, _kw in (('REAL', dict()), ('SHUFFLE', dict(perm=shuffle_perm(SHUFFLE_SEED))),
                   ('NULL', dict(force_null=True))):
    _P, _GS = predict({'B1': _m}, 'val', BATCH*2, **_kw)
    CTRL[_mode] = dict(**eval_all(_P['B1'], 'val'), gate=_GS['B1'], _p=_P['B1'])
ctab = pd.DataFrame([{'mode': k, **{c[1]: CTRL[k][c[0]] for c in COLS},
                      'gate_mean': CTRL[k]['gate']['gate_mean'],
                      'res/|z_img|': CTRL[k]['gate']['res_over_zimg']} for k in CTRL])
print(f"=== CONTROL tai checkpoint B1 ep{SEL['B1']['epoch']} (chi eval) ===")
print(ctab.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
ctab.to_csv(f'{SC_DIR}/metrics/control_real_shuffle_null.csv', index=False)
for _k in ('macro_map','rare3_map'):
    print(f"  {_k:<12} REAL-SHUFFLE {CTRL['REAL'][_k]-CTRL['SHUFFLE'][_k]:+.4f} | "
          f"REAL-NULL {CTRL['REAL'][_k]-CTRL['NULL'][_k]:+.4f}")

_PA, _ = predict({'A1': (lambda m: (m.load_state_dict(torch.load(
            f"{NB14_DIR}/snapshots/A1_ep{SEL['A1']['epoch']}.pt", map_location=dev, weights_only=False)),
            m.eval())[1])(DiseaseGateModel(True).to(dev))}, 'val', BATCH*2) if 'A1' in SEL else (None, None)
if _PA is not None:
    BOOT = paired_bootstrap(_PA['A1'], CTRL['REAL']['_p'], 'val', n_boot=1000, seed=0)
    print('\n=== PAIRED BOOTSTRAP  delta = B1 - A1  (95% CI, 1000 resample) ===')
    for k, v in BOOT.items(): print(f"  {k:<18} {v['mean']:+.4f}  [{v['lo']:+.4f}, {v['hi']:+.4f}]")
    json.dump(BOOT, open(f'{SC_DIR}/metrics/bootstrap_B1_vs_A1.json','w'), indent=1)
else:
    BOOT = {}

d_r3  = S('B1','rare3_map')  - S('A1','rare3_map')
d_mac = S('B1','macro_map')  - S('A1','macro_map')
d_nt  = S('B1','nontarget10_map') - S('A1','nontarget10_map')
_pa, _pb = np.array(SEL['A1']['metrics']['per_ap']), np.array(SEL['B1']['metrics']['per_ap'])
n_imp = int((( _pb - _pa)[RARE3_IDX] > 0).sum())
ok = (d_r3 >= 0.002) and (d_mac >= -0.002) and (d_nt >= -0.0015) and (n_imp >= 2)
print('\n=== GO / NO-GO — B1 vs A1 (cung kien truc, chi doi loss) ===\n')
print(f'  dRare-3        {d_r3:+.4f}  (>= +0.002)')
print(f'  dMacro         {d_mac:+.4f}  (>= -0.002)')
print(f'  dNonTarget-10  {d_nt:+.4f}  (>= -0.0015)')
print(f'  Rare-3 cai thien {n_imp}/3 nhan  (>= 2)')
for l, j in zip(RARE3, RARE3_IDX): print(f'     {l:<16} {_pa[j]:.4f} -> {_pb[j]:.4f}  ({_pb[j]-_pa[j]:+.4f})')
print('\n' + ('KET LUAN: B1 PASS -> can nhac B2 (selective WASL, lambda_W = 3/13) va replicate 3 seed.'
      if ok else
      'KET LUAN: B1 KHONG PASS cong.\n'
      '  Van doc phan STABILITY: neu sut peak->cuoi nho hon han nb12 thi kien truc image-fallback\n'
      '  VAN co gia tri (on dinh hon), du khong tang duoc Rare-3.'))

json.dump(dict(selected={a: dict(epoch=SEL[a]['epoch'], **{k: SEL[a]['metrics'][k] for k in METRIC_KEYS})
                         for a in ORDER},
               factorial=fac.to_dict('records'), stability=stab.to_dict('records'),
               control={k: {c[0]: CTRL[k][c[0]] for c in COLS} for k in CTRL},
               bootstrap=BOOT, nb12_wasl_ref=NB12_WASL,
               delta_vs_A1=dict(rare3=d_r3, macro=d_mac, nontarget10=d_nt, n_rare3_improved=n_imp),
               passed=bool(ok), deviation=PREREG['deviation']),
          open(f'{SC_DIR}/metrics/stageB_result.json','w'), indent=1)
print('\nsaved -> metrics/stageB_result.json, main_table.csv, factorial.csv, stability.csv, '
      'control_real_shuffle_null.csv, bootstrap_B1_vs_A1.json')
print('\nSTOP.')